# 01. Dataset Curation, Label Distribution & ESM-2 Embeddings

## 1. Problem Overview
Predicting the **subcellular localization** of a protein from its amino acid sequence is fundamental to understanding its biological function, interaction partners, and potential drug target viability. 

This project frames subcellular localization as a **multi-label classification** task across **11 major cellular compartments** defined in the DeepLoc 2.0 benchmark:
- *Membrane, Cytoplasm, Nucleus, Extracellular, Cell membrane, Mitochondrion, Plastid, Endoplasmic reticulum, Lysosome/Vacuole, Golgi apparatus, Peroxisome*.

Rather than relying on evolutionary Multiple Sequence Alignments (MSAs) or handcrafted physicochemical features, we leverage frozen representations from **ESM-2** (`esm2_t33_650M_UR50D`, a 650M-parameter protein language model).

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import protein_loc as pl

%matplotlib inline
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'

## 2. Load Curated Datasets

We load the Swiss-Prot development dataset with precomputed 1280-dimensional ESM-2 embeddings (mean pooling) and the independent Human Protein Atlas (HPA) test set.

In [ ]:
train_df, X_mean, y_train = pl.load_subcellular_data('../data/df_train_loc_mean.csv')
_, X_max, _ = pl.load_subcellular_data('../data/df_train_loc_max.csv')
test_df, X_test, y_test = pl.load_subcellular_data('../data/df_test_loc_mean.csv', is_test=True)

print(f"Development set: {X_mean.shape[0]:,} proteins, {X_mean.shape[1]} embedding features")
print(f"Independent HPA test set: {X_test.shape[0]:,} proteins, {X_test.shape[1]} embedding features")
print(f"Label space: {len(pl.SUBCELLULAR_LOCATIONS)} target compartments")

## 3. Label Distribution & Class Imbalance

Subcellular compartments display significant natural imbalance. For instance, `Membrane` and `Cytoplasm` are heavily populated, whereas `Peroxisome` and `Plastid` are comparatively rare.

In [ ]:
counts = pd.Series(y_train.sum(axis=0), index=pl.SUBCELLULAR_LOCATIONS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=150)
bars = ax.bar(counts.index, counts.values, color='#1f77b4', edgecolor='#0f4c81', alpha=0.85)
ax.set_ylabel('Number of Annotated Proteins', fontsize=11)
ax.set_title('Subcellular Compartment Class Distribution (Swiss-Prot Dev Set)', fontsize=12, pad=12)
plt.xticks(rotation=35, ha='right', fontsize=9.5)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=8.5)

plt.tight_layout()
plt.show()

## 4. Multi-Localization Cardinality

Many proteins localize to multiple cellular compartments simultaneously (e.g. shuttling between Nucleus and Cytoplasm). The multi-label nature of this problem is central to our evaluation.

In [ ]:
cardinality = y_train.sum(axis=1)
unique_k, counts_k = np.unique(cardinality, return_counts=True)

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
ax.bar(unique_k, counts_k, color='#2ca02c', edgecolor='#1b611b', alpha=0.85, width=0.6)
ax.set_xlabel('Number of Localizations per Protein (k)', fontsize=11)
ax.set_ylabel('Number of Proteins', fontsize=11)
ax.set_title('Localization Cardinality Distribution', fontsize=12, pad=12)
ax.set_xticks(unique_k)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for x, y_val in zip(unique_k, counts_k):
    pct = (y_val / len(cardinality)) * 100
    ax.annotate(f'{y_val} ({pct:.1f}%)', (x, y_val), textcoords="offset points", xytext=(0, 3), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 5. ESM-2 Embeddings: Mean vs. Max Pooling Projections

We compute 2D Principal Component Analysis (PCA) and t-SNE projections of the 1280-dimensional embeddings to inspect latent spatial clustering.

In [ ]:
# Sample a subset for clean 2D visualization
np.random.seed(42)
sample_idx = np.random.choice(len(X_mean), size=2000, replace=False)
X_sample = X_mean[sample_idx]
y_sample = y_train[sample_idx]

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sample)

# Color by primary compartment (first positive class or Multi-localized)
primary_label = []
for row in y_sample:
    pos = np.where(row == 1)[0]
    if len(pos) == 0:
        primary_label.append('None')
    elif len(pos) > 1:
        primary_label.append('Multi-localized')
    else:
        primary_label.append(pl.SUBCELLULAR_LOCATIONS[pos[0]])

primary_series = pd.Series(primary_label)
top_classes = ['Multi-localized', 'Membrane', 'Cytoplasm', 'Nucleus', 'Mitochondrion', 'Extracellular']

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
for cls in top_classes:
    mask = (primary_series == cls).values
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=cls, alpha=0.6, s=16)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=10)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=10)
ax.set_title('PCA of ESM-2 (650M) Protein Sequence Embeddings', fontsize=12, pad=12)
ax.legend(bbox_to_anchor=(1.04, 1), loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()